# LSTM Forecasting

## 1. Import Libraries

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, LSTM
from tensorflow.keras.models import Sequential

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_bitcoin_data
from src.preprocessing import prepare_daily_bitcoin_data
from src.metrics import mae, rmse, mape, smape

tf.random.set_seed(42)
np.random.seed(42)

ModuleNotFoundError: No module named 'tensorflow'

## 2. Load Dataset

In [ ]:
data_path = PROJECT_ROOT / "data" / "bitcoin" / "btcusd_1-min_data.csv"

df_raw = load_bitcoin_data(data_path)
df_daily = prepare_daily_bitcoin_data(df_raw)
target = df_daily["Close"].dropna().asfreq("D")

df_daily.head()

## 3. Train-Test Split

In [ ]:
split_idx = int(len(target) * 0.8)

train = target.iloc[:split_idx]
test = target.iloc[split_idx:]

train.shape, test.shape

## 4. Data Scaling

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))

train_values = train.to_numpy().reshape(-1, 1)
test_values = test.to_numpy().reshape(-1, 1)

train_scaled = scaler.fit_transform(train_values)
test_scaled = scaler.transform(test_values)

## 5. Sequence Creation

In [ ]:
LOOKBACK = 30


def create_sequences(values, lookback):
    X, y = [], []
    for i in range(lookback, len(values)):
        X.append(values[i - lookback : i])
        y.append(values[i])
    return np.array(X), np.array(y)


X_train, y_train = create_sequences(train_scaled, LOOKBACK)

combined_scaled = np.vstack([train_scaled[-LOOKBACK:], test_scaled])
X_test, y_test_scaled = create_sequences(combined_scaled, LOOKBACK)
y_test = test.copy()

X_train.shape, y_train.shape, X_test.shape, y_test.shape

## 6. Build LSTM Model

In [ ]:
model = Sequential(
    [
        LSTM(32, input_shape=(LOOKBACK, 1)),
        Dense(1),
    ]
)

model.compile(optimizer="adam", loss="mse")
model.summary()

## 7. Train Model

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
)

history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stopping],
    shuffle=False,
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history.history["loss"], label="Train Loss")
ax.plot(history.history["val_loss"], label="Validation Loss")
ax.set_title("LSTM Training Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 8. Generate Forecasts

In [ ]:
lstm_predictions_scaled = model.predict(X_test)
lstm_predictions = scaler.inverse_transform(lstm_predictions_scaled).ravel()
lstm_forecast = pd.Series(lstm_predictions, index=test.index, name="LSTM")

fig, ax = plt.subplots(figsize=(12, 5))
train.tail(180).plot(ax=ax, label="Train")
test.plot(ax=ax, label="Test")
lstm_forecast.plot(ax=ax, label="LSTM Forecast")
ax.set_title("Bitcoin Close Price: LSTM Forecast")
ax.set_xlabel("Date")
ax.set_ylabel("Close")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 9. Evaluation Metrics

In [ ]:
def evaluate_forecast(y_true, y_pred):
    return {
        "MAE": mae(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "MAPE": mape(y_true, y_pred),
        "sMAPE": smape(y_true, y_pred),
    }


lstm_metrics = pd.DataFrame(
    [evaluate_forecast(y_test, lstm_forecast)],
    index=["LSTM"],
)

lstm_metrics

## 10. Compare with Classical Benchmark

In [ ]:
naive_forecast = target.shift(1).reindex(test.index).rename("Naive")
moving_average_forecast = (
    target.shift(1)
    .rolling(window=7)
    .mean()
    .reindex(test.index)
    .rename("7-Day Moving Average")
)

benchmark_forecasts = {
    "Naive": naive_forecast,
    "7-Day Moving Average": moving_average_forecast,
    "LSTM": lstm_forecast,
}

comparison_table = pd.DataFrame(
    [evaluate_forecast(y_test, forecast) for forecast in benchmark_forecasts.values()],
    index=benchmark_forecasts.keys(),
)

fig, ax = plt.subplots(figsize=(12, 5))
y_test.plot(ax=ax, label="Actual", linewidth=2)

for model_name, forecast in benchmark_forecasts.items():
    forecast.plot(ax=ax, label=model_name, alpha=0.85)

ax.set_title("LSTM vs Classical Forecast Benchmarks")
ax.set_xlabel("Date")
ax.set_ylabel("Close")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

comparison_table.sort_values("RMSE")

## 11. Key Findings
- This notebook starts with a simple single-layer LSTM using a 30-day lookback window.
- The LSTM forecast is evaluated on the same chronological test period as the classical baselines.
- Naive and 7-day moving average forecasts provide lightweight benchmarks for judging whether the LSTM adds value.
- Lower MAE, RMSE, MAPE, and sMAPE values indicate stronger out-of-sample performance.